In [ ]:
# Exercício **B.6.20**

_Diagrama (Ogata, Figura 6.110): malha unitária negativa com **compensador de atraso de fase** \(G_c(s)\) em série com o sistema de **posicionamento angular** \(G(s)=\dfrac{820}{s(s+10)(s+20)}\)._

# Solução completa — **B.6.20**

> **Figura:** veja a **Figura 6.110** no livro.

## Enunciado

O sistema em malha fechada apresenta polos dominantes em **\(s=-3{,}60\pm j4{,}80\)**, com **\(\zeta=0{,}6\)** e **\(K_v=4{,}1\ \mathrm{s}^{-1}\)**. Para rampa de **\(360^\circ/\mathrm{s}\)**,

$$
e_v=\frac{\theta_i}{K_v}=\frac{360}{4{,}1}\approx 87{,}8^\circ.
$$

**Preciso** reduzir \(e_v\) para **um décimo** (\(K_v=41\ \mathrm{s}^{-1}\)), **mantendo** **\(\zeta\approx 0{,}6\)** (pequena alteração em \(\omega_n\) é permitida), projetando um **compensador de atraso**.

---

## Passo 1 — Dados do sistema sem compensação

Planta (ganho já ajustado para \(\zeta=0{,}6\)):

$$
G(s)=\frac{820}{s(s+10)(s+20)}.
$$

Do par dominante:

$$
\omega_n=\sqrt{3{,}6^2+4{,}8^2}=6\ \mathrm{rad/s},
\qquad
\zeta=\frac{3{,}6}{6}=0{,}6.
$$

Constante de velocidade estática (sistema tipo 1):

$$
K_v=\lim_{s\to 0} sG(s)=\frac{820}{10\cdot 20}=4{,}1\ \mathrm{s}^{-1}.
$$

---

## Passo 2 — Fator \(\beta\) do compensador de atraso

Para reduzir o erro em rampa por um fator 10:

$$
\beta=\frac{K_{v,\mathrm{desejado}}}{K_{v,\mathrm{atual}}}=\frac{41}{4{,}1}=10.
$$

Modelo (Ogata, Eq. 6-19):

$$
G_c(s)=\hat K_c\,\beta\,\frac{Ts+1}{\beta Ts+1}
=\hat K_c\,\frac{s+\dfrac{1}{T}}{s+\dfrac{1}{\beta T}},
\qquad \beta>1.
$$

O polo fica **mais próximo da origem** que o zero; em regime DC o ganho do compensador é \(\approx \hat K_c\), e \(K_v\) aumenta por \(\beta\) quando \(\hat K_c\approx 1\).

---

## Passo 3 — Escolha de zero e polo (próximos da origem)

Seguindo o **Exemplo 6-7** de Ogata (mesmo \(\beta=10\)), coloco zero e polo **próximos e com razão 10**:

$$
s_z=-0{,}05,\qquad s_p=-0{,}005
\quad\Longrightarrow\quad
T=20\ \mathrm{s},\ \ \beta T=200\ \mathrm{s}.
$$

Em \(s_d=-3{,}6+j4{,}8\), o atraso de fase do compensador fica **\(\approx 4^\circ\)** (pequeno), de modo que os polos dominantes se deslocam pouco na reta \(\zeta=0{,}6\).

---

## Passo 4 — Compensador e malha compensada

Com \(\hat K_c\approx 1\):

$$
G_c(s)=\frac{s+0{,}05}{s+0{,}005}.
$$

Malha aberta compensada:

$$
G_{\mathrm{ol}}(s)=G_c(s)\,G(s)
=\frac{s+0{,}05}{s+0{,}005}\cdot\frac{820}{s(s+10)(s+20)}.
$$

**Resultado numérico** (verificação no código):

| Grandeza | Antes | Depois |
|----------|-------|--------|
| \(K_v\) | 4,1 | **41** |
| \(e_v\) (rampa 360°/s) | 87,8° | **≈ 8,78°** |
| Polos dominantes | \(-3{,}60\pm j4{,}80\) | **≈ \(-3{,}58\pm j4{,}77\)** |
| \(\omega_n\) | 6,0 | **≈ 5,96** rad/s |
| \(\zeta\) | 0,60 | **≈ 0,60** |

Aparece ainda um **polo real lento** perto da origem (efeito típico do atraso), o que pode alongar levemente o transitório.

---

## Passo 5 — Verificação e gráficos

**Confiro** polos, \(K_v\), erro em rampa, **lugar das raízes** e **resposta ao degrau** da malha compensada.

In [ ]:
"""
Exercício B.6.20 — compensador de atraso (lag) para posicionamento angular.
Aumento de Kv de 4,1 para 41 s⁻¹ mantendo ζ ≈ 0,6.
"""

from __future__ import annotations

import warnings

import control as ct
import matplotlib.pyplot as plt
import numpy as np

warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    module=r"control\.rlocus",
)

plt.rcParams.update(
    {
        "figure.figsize": (10, 6),
        "figure.dpi": 110,
        "font.size": 11,
        "axes.grid": True,
        "grid.linestyle": "--",
        "grid.alpha": 0.35,
        "axes.labelsize": 12,
        "axes.titlesize": 13,
        "legend.fontsize": 10,
    }
)

# Especificações do enunciado
ZETA_REF = 0.6
KV_ANTES = 4.1
KV_DEPOIS = 41.0
BETA = KV_DEPOIS / KV_ANTES
RAMP_DEG = 360.0

# Polos dominantes originais (referência)
SD_REF = -3.6 + 4.8j


def planta(k: float = 1.0) -> ct.TransferFunction:
    """G(s) = K·820 / [s(s+10)(s+20)]."""
    return k * ct.tf([820.0], [1.0, 30.0, 200.0, 0.0])


def compensador_atraso(
    k_c: float, zero: float, polo: float
) -> ct.TransferFunction:
    """
    Compensador de atraso: Gc(s) = K̂c·(s+zero)/(s+polo), com |zero| > |polo|.

    Equivalente a K̂c·β·(Ts+1)/(βTs+1) com zero = 1/T e polo = 1/(βT).
    """
    return ct.tf(k_c * np.array([1.0, zero]), np.array([1.0, polo]))


def malha_aberta(k_planta: float, k_c: float, zero: float, polo: float) -> ct.TransferFunction:
    """G_ol = Gc·G."""
    return compensador_atraso(k_c, zero, polo) * planta(k_planta)


def malha_fechada(k_planta: float, k_c: float, zero: float, polo: float) -> ct.TransferFunction:
    """Realimentação unitária negativa."""
    return ct.feedback(malha_aberta(k_planta, k_c, zero, polo), 1)


def kv_estatico(sys_ol: ct.TransferFunction) -> float:
    """Kv = lim_{s→0} s·G_ol(s) para sistema tipo 1."""
    # Avaliação numérica próxima da origem (evita divisão simbólica).
    s = 1e-6
    return float(abs(s * ct.evalfr(sys_ol, s)))


def erro_rampa_graus(kv: float, rampa: float = RAMP_DEG) -> float:
    """e_v = θ_i / Kv (entrada em graus/s, Kv em s⁻¹ → erro em graus)."""
    return rampa / kv


def parametros_ogata_exemplo_67() -> tuple[float, float, float, float]:
    """
    Zero/polo do Exemplo 6-7 (β=10): sz=-0,05, sp=-0,005, T=20 s.

    Retorna (zero, polo, T, beta).
    """
    zero = 0.05
    polo = 0.005
    t_val = 1.0 / zero
    return zero, polo, t_val, BETA


def fase_compensador(s: complex, zero: float, polo: float) -> float:
    """Fase de (s+zero)/(s+polo) em graus."""
    return float(np.degrees(np.angle((s + zero) / (s + polo))))


def formatar_polos(polos: np.ndarray) -> None:
    """Lista polos com ζ e ωn."""
    polos = np.sort_complex(polos)
    print(" Polos em malha fechada:")
    for i, p in enumerate(polos, start=1):
        mag = abs(p)
        zeta_p = float(-np.real(p) / mag) if mag > 1e-12 else float("nan")
        print(
            f"   p{i}: {np.real(p):+.6f} {np.imag(p):+.6f} j   "
            f"|s|={mag:.6f}   ζ≈{zeta_p:.6f}"
        )


def relatorio(titulo: str, k_c: float, zero: float, polo: float) -> None:
    """Imprime comparação antes/depois."""
    sep = "=" * 72
    sys_ol = malha_aberta(1.0, k_c, zero, polo)
    sys_cl = malha_fechada(1.0, k_c, zero, polo)
    kv = kv_estatico(sys_ol)
    ev = erro_rampa_graus(kv)

    print(sep)
    print(f" {titulo}")
    print(sep)
    print(f" β = Kv_novo/Kv_antigo = {BETA:g}")
    print(f" Zero do compensador: s = {-zero:.6g}   (T = {1/zero:.6g} s)")
    print(f" Polo do compensador: s = {-polo:.6g}   (βT = {1/polo:.6g} s)")
    print(f" K̂c ≈ {k_c:.6g}")
    print(f" Fase de Gc em s_ref: {fase_compensador(SD_REF, zero, polo):.4f}°")
    print()
    print(f" Kv ≈ {kv:.6g} s⁻¹   (meta: {KV_DEPOIS:g})")
    print(f" e_v (rampa {RAMP_DEG:g}°/s) ≈ {ev:.4f}°   (antes: {erro_rampa_graus(KV_ANTES):.4f}°)")
    formatar_polos(ct.poles(sys_cl))
    print(sep)


def plot_lugar_raizes(zero: float, polo: float) -> None:
    """Compara RL sem e com compensador de atraso."""
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

    for ax, sys_ol, titulo in [
        (axes[0], planta(1.0), "Sem compensador"),
        (axes[1], malha_aberta(1.0, 1.0, zero, polo), "Com compensador de atraso"),
    ]:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=UserWarning)
            ct.root_locus_plot(sys_ol, plot=True, grid=True, ax=ax)
        polos_mf = ct.poles(ct.feedback(sys_ol, 1))
        ax.scatter(
            np.real(polos_mf),
            np.imag(polos_mf),
            s=85,
            c="crimson",
            marker="x",
            linewidths=2.0,
            label="Polos MF (K=1)",
            zorder=5,
        )
        ax.scatter(
            [-3.6],
            [4.8],
            s=55,
            c="#ff7f0e",
            marker="o",
            label="Dominante ref.",
            zorder=6,
        )
        ax.set_title(titulo)
        ax.set_xlabel(r"$\mathrm{Re}(s)$")
        ax.set_ylabel(r"$\mathrm{Im}(s)$")
        ax.legend(loc="best", fontsize=9)

    fig.suptitle(r"Lugar das raízes — $G(s)=\dfrac{820}{s(s+10)(s+20)}$", fontsize=13)
    plt.tight_layout()


def plot_degrau(k_c: float, zero: float, polo: float) -> None:
    """Resposta ao degrau da malha compensada."""
    sys_cl = malha_fechada(1.0, k_c, zero, polo)
    t = np.linspace(0.0, 3.0, 3000)
    tout, yout = ct.step_response(sys_cl, t)
    fig, ax = plt.subplots()
    ax.plot(tout, yout, lw=2.0, label=r"$y(t)$ — com atraso")
    sys_ref = ct.feedback(planta(1.0), 1)
    _, y0 = ct.step_response(sys_ref, t)
    ax.plot(tout, y0, ls="--", color="gray", lw=1.5, label="Sem compensador")
    ax.set_title("Resposta ao degrau unitário")
    ax.set_xlabel(r"$t$ (s)")
    ax.set_ylabel(r"$y(t)$")
    ax.legend(loc="best")
    plt.tight_layout()


# --- Parâmetros do compensador (Ogata, Ex. 6-7) ---
ZERO, POLO, T_LAG, BETA_CHK = parametros_ogata_exemplo_67()
K_C = 1.0

relatorio("SOLUÇÃO — compensador de atraso (β = 10)", K_C, ZERO, POLO)

plot_lugar_raizes(ZERO, POLO)
plt.show()

plot_degrau(K_C, ZERO, POLO)
plt.show()